In [1]:
# ===== 1) Kurulum & Model =====
import os, math
import numpy as np
import pandas as pd
import torch
import cv2
from tqdm import tqdm

from groundingdino.util.inference import load_model, load_image, predict

# --- Yollar ---
CSV_PATH      = r"C:\Users\selam\Desktop\kod deneme\gd\veriler\cat_food.csv"
OUTPUT_DIR    = r"C:\Users\selam\Desktop\kod deneme\gd\veriler\outputs"
CONFIG_PATH   = r"C:\Users\selam\Desktop\grounding-dino-base\groundingdino\config\GroundingDINO_SwinB_cfg.py"
WEIGHTS_PATH  = r"C:\Users\selam\Desktop\grounding-dino-base\weights\groundingdino_swinb_cogcoor.pth"

os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

model = load_model(CONFIG_PATH, WEIGHTS_PATH, device=DEVICE)
print("GroundingDINO yüklendi.")


Device: cuda


final text_encoder_type: bert-base-uncased
GroundingDINO yüklendi.


In [2]:
# ===== 2) Veri =====
df = pd.read_csv(CSV_PATH)
print("Toplam satır:", len(df))
df.head()


Toplam satır: 93


,groupID_s,category1Name,category1Name_en,category2Name,category2Name_en,category3Name,category3Name_en,category4Name,category4Name_en,imagePath_s
0,287863,Evcil Hayvan Ürünleri,Pet Products,Kedi,Cat,Kedi Maması,Cat Food,Kedi Kuru Mama,Cat Dry Food,C:\Users\selam\Desktop\kod deneme\gd\veriler\i...
1,1480242,Evcil Hayvan Ürünleri,Pet Products,Kedi,Cat,Kedi Maması,Cat Food,Kedi Kuru Mama,Cat Dry Food,C:\Users\selam\Desktop\kod deneme\gd\veriler\i...
2,315286,Evcil Hayvan Ürünleri,Pet Products,Kedi,Cat,Kedi Maması,Cat Food,Kedi Kuru Mama,Cat Dry Food,C:\Users\selam\Desktop\kod deneme\gd\veriler\i...
3,1129722,Evcil Hayvan Ürünleri,Pet Products,Kedi,Cat,Kedi Maması,Cat Food,Kedi Kuru Mama,Cat Dry Food,C:\Users\selam\Desktop\kod deneme\gd\veriler\i...
4,1131897,Evcil Hayvan Ürünleri,Pet Products,Kedi,Cat,Kedi Maması,Cat Food,Kedi Kuru Mama,Cat Dry Food,C:\Users\selam\Desktop\kod deneme\gd\veriler\i...


In [40]:
# === Cell 3 ===
import torch, math, numpy as np
from datetime import datetime

def pick_prompt(row):
   for col in ['category4Name_en', 'category3Name_en', 'category2Name_en', 'category1Name_en']:
       if col in row and pd.notna(row[col]) and str(row[col]).strip():
           return str(row[col]).strip()
   return ""

def compute_confidences(logits_tensor):
   t = logits_tensor.detach().float().flatten()
   if t.min().item() >= -1e-6 and t.max().item() <= 1.000001:
       probs = t.clamp(0, 1)
   else:
       probs = torch.sigmoid(t)
   return probs.cpu().numpy()

def clamp01(x): 
   """Bir sayıyı 0-1 aralığında sınırlar"""
   return max(0.0, min(1.0, float(x)))

def box_cxcywh_norm_to_xyxy_pixels(b, W, H):
   cx, cy, w, h = map(float, b)
   x1 = (cx - w/2.0) * W
   y1 = (cy - h/2.0) * H
   x2 = (cx + w/2.0) * W
   y2 = (cy + h/2.0) * H
   x1, x2 = sorted([max(0, round(x1)), min(W, round(x2))])
   y1, y2 = sorted([max(0, round(y1)), min(H, round(y2))])
   return int(x1), int(y1), int(x2), int(y2)

def balanced_score(p, a_norm, W_P, W_S, min_area_pct=0.10):
   """
   Basit dengeli skor: Parametreli ağırlık + minimum alan kontrolü
   - Alan min_area_pct'den büyükse: W_P * p + W_S * area
   - Alan çok küçükse: 0 (elenir)
   """
   p_clean = clamp01(p)
   a_clean = clamp01(a_norm)
   
   # Minimum alan kontrolü
   if a_clean >= min_area_pct:
       score = W_P * p_clean + W_S * a_clean
   else:
       score = 0  # Çok küçük alanları ele
   
   return score

In [41]:
def process_one(image_path, prompt, tmp_out_path,
                box_threshold, text_threshold, W_P, W_S):
    """
    - Her kutu için: p, alan → balanced_score
    - En yüksek skorlu kutuyu kırp
    """
    info = {
        "num_boxes": 0,
        "valid_boxes": 0,
        "chosen": None,
        "top3": [],
        "weights": (W_P, W_S),
        "thresholds": (box_threshold, text_threshold)
    }

    bgr = cv2.imread(image_path)
    if bgr is None:
        raise FileNotFoundError(f"Görsel yüklenemedi: {image_path}")
    H, W = bgr.shape[:2]
    total_area = W * H

    _, image_tensor = load_image(image_path)
    boxes, logits, phrases = predict(
        model=model,
        image=image_tensor,
        caption=prompt,
        box_threshold=box_threshold,
        text_threshold=text_threshold,
        device=DEVICE
    )

    N = len(boxes)
    info["num_boxes"] = int(N)
    if N == 0:
        cv2.imwrite(tmp_out_path, bgr)
        return info

    probs = compute_confidences(logits)

    candidates = []
    for k in range(N):
        cx, cy, w, h = map(float, boxes[k])
        a_norm = clamp01(w * h)
        p = float(probs[k])
        S = balanced_score(p, a_norm, W_P, W_S)
        
        # Sadece geçerli skorlu kutuları (alan ≥ 0.1) candidates'a ekle
        if S > 0:
            x1, y1, x2, y2 = box_cxcywh_norm_to_xyxy_pixels(boxes[k], W, H)
            crop_area = (x2 - x1) * (y2 - y1)
            area_percentage = (crop_area / total_area) * 100
            
            candidates.append({
                "idx": k, "S": S, "p": p, "a_norm": a_norm,
                "coords": (x1, y1, x2, y2),
                "area_percentage": area_percentage
            })

    info["valid_boxes"] = len(candidates)
    if len(candidates) == 0:
        cv2.imwrite(tmp_out_path, bgr)
        return info

    candidates.sort(key=lambda d: d["S"], reverse=True)
    info["top3"] = [{
        "idx": c["idx"], "S": round(c["S"], 4), "p": round(c["p"], 4),
        "a": round(c["a_norm"], 4), "area_pct": round(c["area_percentage"], 2),
        "coords": c["coords"]
    } for c in candidates[:3]]

    best = candidates[0]
    x1, y1, x2, y2 = best["coords"]
    info["chosen"] = best

    crop = bgr[y1:y2, x1:x2]
    if crop.size == 0:
        cv2.imwrite(tmp_out_path, bgr)
        return info

    cv2.imwrite(tmp_out_path, crop)
    return info

In [42]:
# === Cell 5 ===
import os, time
from datetime import timedelta, datetime

# Ayarlayacağın parametreler
BOX_THRESHOLD  = 0.1
TEXT_THRESHOLD = 0.1

# Ağırlıklar: confidence ve alan
W_P, W_S = 0.3, 0.7

# Log dosyası yolu
log_path = r"C:\Users\selam\Desktop\kod deneme\gd\veriler\outputs_log.txt"

def log_to_file(message: str):
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(message + "\n")

total = len(df)
start_t = time.time()
processed = 0

for i, row in df.iterrows():
    gid    = row['groupID_s']
    prompt = pick_prompt(row)
    img_p  = str(row['imagePath_s']).strip()
    out_path = os.path.join(OUTPUT_DIR, f"{gid}.jpg")

    try:
        info = process_one(
            image_path   = img_p,
            prompt       = prompt,
            tmp_out_path = out_path,
            box_threshold= BOX_THRESHOLD,
            text_threshold= TEXT_THRESHOLD,
            W_P=W_P, W_S=W_S
        )

        # --- LOG ---
        current_time = datetime.now().strftime("%H:%M:%S")
        log_to_file("\n" + "="*70 + "\n")
        log_to_file(f"[{i+1}/{total}] | {current_time} | ID={gid} | Prompt='{prompt}'")
        log_to_file(f"  -> Toplam kutu: {info['num_boxes']} | Geçerli kutu (area≥10%): {info['valid_boxes']}")

        if info['valid_boxes'] > 0:
            log_to_file("  -> Top-3 Sıralaması (idx / S / p / area_pct / coords):")
            for item in info["top3"]:
                idx  = item.get("idx")
                s    = float(item.get("S", 0))
                p    = float(item.get("p", 0))
                area_pct = float(item.get("area_pct", 0))
                x1, y1, x2, y2 = item.get("coords", (None, None, None, None))
                log_to_file(
                    f"     idx={idx} | S={s:.4f} | p={p:.4f} | "
                    f"area_pct={area_pct:.2f}% | coords=({x1}, {y1}, {x2}, {y2})"
                )

        processed += 1
        elapsed = time.time() - start_t
        rate = processed / elapsed if elapsed > 0 else 0.0
        remaining = (total - processed) / rate if rate > 0 else 0.0
        print(
            f"\rProcessed {processed}/{total} | "
            f"Elapsed: {str(timedelta(seconds=int(elapsed)))} | "
            f"ETA: {str(timedelta(seconds=int(remaining)))}",
            end="", flush=True
        )

    except Exception:
        processed += 1
        elapsed = time.time() - start_t
        rate = processed / elapsed if elapsed > 0 else 0.0
        remaining = (total - processed) / rate if rate > 0 else 0.0
        print(
            f"\rProcessed {processed}/{total} | "
            f"Elapsed: {str(timedelta(seconds=int(elapsed)))} | "
            f"ETA: {str(timedelta(seconds=int(remaining)))}",
            end="", flush=True
        )

print()

Processed 93/93 | Elapsed: 0:01:26 | ETA: 0:00:00
